# SOTA Sentiment Classification using RoBERTa

This notebook classifies the `cleaned_review` column using:
`cardiffnlp/twitter-roberta-base-sentiment-latest`

It outputs:
- `pred_subjectivity` (1 opinion, 0 factual/neutral)
- `pred_polarity` (1 positive, -1 negative, 0 neutral)

and saves `full_dataset_with_predictions.csv`.

In [ ]:
# ------------------------------
# Imports (install torch, transformers, tqdm in your environment if needed)
# ------------------------------
import pandas as pd
import torch
from tqdm.auto import tqdm

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

In [2]:
# ------------------------------
# 1) Load data (use cleaned text)
# ------------------------------
INPUT_CSV = "hawker_corpus_final10k_cleaned.csv"
OUTPUT_CSV = "full_dataset_with_predictions.csv"

df = pd.read_csv(INPUT_CSV)
if "cleaned_review" not in df.columns:
    raise ValueError(f"Expected column `cleaned_review` but got: {df.columns.tolist()}")

texts = df["cleaned_review"].fillna("").astype(str).tolist()
n = len(texts)
print(f"Loaded {n} reviews")

Loaded 11032 reviews


In [3]:
# ------------------------------
# 2) Model + label mapping
# ------------------------------
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

# id2label comes from model config
id2label = model.config.id2label
print("Model id2label:", id2label)

label_to_index = {label.lower(): idx for idx, label in id2label.items()}
positive_idx = label_to_index.get("positive")
neutral_idx = label_to_index.get("neutral")
negative_idx = label_to_index.get("negative")

if positive_idx is None or neutral_idx is None or negative_idx is None:
    raise ValueError("Could not map model labels to Positive/Neutral/Negative. Please check id2label.")

def polarity_from_pred_idx(pred_idx: int) -> int:
    if pred_idx == positive_idx:
        return 1
    if pred_idx == negative_idx:
        return -1
    return 0

# Subjectivity threshold for neutral confidence
NEUTRAL_CONF_THRESHOLD = 0.5
print("Neutral confidence threshold:", NEUTRAL_CONF_THRESHOLD)

Using device: cpu


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 34411.82it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model id2label: {0: 'negative', 1: 'neutral', 2: 'positive'}
Neutral confidence threshold: 0.5


In [4]:
# ------------------------------
# 3) Batched inference with tqdm
# ------------------------------
batch_size = 16  # increase to 32 if your machine can handle it
max_length = 256 # adjust if you want more context (slower for longer max_length)

pred_subjectivity = [0] * n
pred_polarity = [0] * n

import time
t0 = time.perf_counter()

softmax = torch.nn.Softmax(dim=-1)

with torch.no_grad():
    for start_idx in tqdm(range(0, n, batch_size), desc="Classifying", unit="batch"):
        end_idx = min(n, start_idx + batch_size)
        batch_texts = texts[start_idx:end_idx]

        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        logits = model(**enc).logits  # [batch, num_labels]
        probs = softmax(logits)       # [batch, num_labels]

        # Extract probs needed by your rules
        neu_probs = probs[:, neutral_idx]

        # Predicted label index = argmax over all labels
        pred_idx = torch.argmax(probs, dim=-1).tolist()

        # Apply mapping rules per row
        for i in range(end_idx - start_idx):
            row = start_idx + i
            p = polarity_from_pred_idx(pred_idx[i])

            # Subjectivity rule:
            # If neutral confidence < threshold OR predicts positive/negative => subjective (1)
            neutral_conf = float(neu_probs[i].item())
            is_subjective = 1 if ((p != 0) or (neutral_conf < NEUTRAL_CONF_THRESHOLD)) else 0

            pred_polarity[row] = p
            pred_subjectivity[row] = is_subjective

elapsed_sec = time.perf_counter() - t0
rps = n / elapsed_sec if elapsed_sec > 0 else float("inf")

print(f"Total time: {elapsed_sec:.3f}s")
print(f"Records per second (RPS): {rps:.2f}")
print("Polarity counts:")
print(pd.Series(pred_polarity).value_counts().sort_index())
print("Subjectivity counts:")
print(pd.Series(pred_subjectivity).value_counts().sort_index())

Classifying: 100%|██████████| 690/690 [12:50<00:00,  1.12s/batch]

Total time: 770.631s
Records per second (RPS): 14.32
Polarity counts:
-1    2225
 0    1120
 1    7687
Name: count, dtype: int64
Subjectivity counts:
0      892
1    10140
Name: count, dtype: int64


In [5]:
# ------------------------------
# 4) Save results
# ------------------------------
df["pred_subjectivity"] = pred_subjectivity
df["pred_polarity"] = pred_polarity

df.to_csv(OUTPUT_CSV, index=False)
print("Saved:", OUTPUT_CSV)

# Quick sanity check: if pred_subjectivity == 0 then polarity must be 0
viol = df[(df["pred_subjectivity"] == 0) & (df["pred_polarity"] != 0)]
print("Rule violations (pred_subjectivity=0 but pred_polarity!=0):", len(viol))
df[["pred_subjectivity", "pred_polarity"]].head(5)

Saved: full_dataset_with_predictions.csv
Rule violations (pred_subjectivity=0 but pred_polarity!=0): 0


,pred_subjectivity,pred_polarity
0,1,1
1,1,1
2,1,1
3,1,1
4,1,1
